# E-Commerce Sales & Profitability Analysis

**Python / Pandas — Data Exploration, Financial Validation, Reconciliation, and Category Analysis**

This notebook documents an investigation of a likely synthetic e-commerce dataset (2021–2025). Its focus is on understanding source tables, validating financial consistency, reconciling order-item records against reported order totals, and preparing U.S. category data for Power BI. The source data’s authenticity has not been independently verified.

### Files required to reproduce
Place the following files in the same folder as this notebook before running: `ecommerce_sales_customer_analytics_150k.csv`, `order_items.csv`, and `product_catalog.csv`. Install `pandas` in your Python environment. The original CSV files are not embedded in this notebook.

**Note:** Financial measures, especially `net_sales` and `profit`, follow dataset-specific formulas and should not be interpreted as audited accounting metrics. Excluded item rows are not proven invalid; the reconciliation aligns item totals to the source order totals.


In [ ]:
import pandas as pd
print (pd.__version__)

3.1.0


In [ ]:
df = pd.read_csv("ecommerce_sales_customer_analytics_150k.csv")

In [ ]:
df.head()

,order_id,order_date,order_time,order_status,sales_channel,customer_id,customer_name,customer_age,gender,customer_segment,...,discount_amount,tax_amount,shipping_cost,net_sales,product_cost,profit,profit_margin_percentage,customer_lifetime_value,is_repeat_customer,customer_order_count
0,ORD-301242,2023-11-06,16:37:47,Completed,Mobile App,CUST-003102,Jasmine Ryan,55,Male,Consumer,...,477.89,61.07,12.03,945.40,588.33,345.04,36.50,12459.68,True,11
1,ORD-773460,2025-12-24,01:22:36,Completed,Website,CUST-003124,Scott Chase,26,Male,Premium,...,1456.06,320.83,9.00,2018.41,2031.88,-22.47,-1.11,13032.48,True,11
2,ORD-449374,2021-07-05,14:24:21,Completed,Mobile App,CUST-012496,Marc Wheeler,69,Female,Premium,...,97.30,29.79,13.05,468.35,257.73,197.57,42.18,6159.44,True,6
3,ORD-567636,2023-01-21,07:20:26,Completed,Social Media,CUST-023928,Jennifer Smith,65,Male,Consumer,...,53.34,34.39,20.85,546.52,277.84,247.83,45.35,5638.30,True,7
4,ORD-820028,2022-05-13,09:46:21,Completed,Social Media,CUST-012730,Jessica Wang,34,Female,Consumer,...,133.63,421.35,23.03,2785.27,1231.88,1530.36,54.94,13240.10,True,8


In [ ]:
df.shape

(138116, 46)

In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 138116 entries, 0 to 138115
Data columns (total 46 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   order_id                  138116 non-null  str    
 1   order_date                138116 non-null  str    
 2   order_time                138116 non-null  str    
 3   order_status              138116 non-null  str    
 4   sales_channel             138116 non-null  str    
 5   customer_id               138116 non-null  str    
 6   customer_name             138116 non-null  str    
 7   customer_age              138116 non-null  int64  
 8   gender                    138116 non-null  str    
 9   customer_segment          138116 non-null  str    
 10  customer_type             138116 non-null  str    
 11  customer_city             138116 non-null  str    
 12  customer_state            138116 non-null  str    
 13  customer_country          138116 non-null  str    
 14 

In [ ]:
df.isnull().sum()

order_id                         0
order_date                       0
order_time                       0
order_status                     0
sales_channel                    0
customer_id                      0
customer_name                    0
customer_age                     0
gender                           0
customer_segment                 0
customer_type                    0
customer_city                    0
customer_state                   0
customer_country                 0
region                           0
customer_postal_code             0
payment_method                   0
payment_status                   0
currency                         0
shipping_method                  0
warehouse                        0
delivery_days                24557
estimated_delivery_days      24557
delivery_status                  0
return_status               128654
return_reason               128654
customer_rating              24557
review_sentiment             24557
customer_review     

In [ ]:
df.loc[
    df["delivery_days"].isna(),
    "order_status"
].value_counts()

order_status
Returned     9462
Cancelled    8398
Pending      6697
Name: count, dtype: int64

In [ ]:
pd.crosstab(
    df["order_status"],
    df["delivery_days"].isna()
)

delivery_days,False,True
order_status,,
Cancelled,0,8398
Completed,113559,0
Pending,0,6697
Returned,0,9462


### Missing Values — Delivery Days

The dataset contains 24,557 missing values in the `delivery_days` column (17.8% of all orders).

All 113,559 completed orders have recorded delivery times, while all cancelled, pending, and returned orders have missing delivery times.

This indicates that missing delivery information is systematically associated with order status.

**Decision:** Keep missing values unchanged. Use completed orders only when analyzing actual delivery times.

**Limitation:** Delivery times for returned orders are unavailable, so their delivery performance cannot be evaluated.

In [ ]:
missing = df.isnull().sum()
missing [missing >0]

delivery_days               24557
estimated_delivery_days     24557
return_status              128654
return_reason              128654
customer_rating             24557
review_sentiment            24557
customer_review             24557
campaign_name               83233
coupon_code                110502
dtype: int64

In [ ]:
pd.crosstab(
    df["order_status"],
    df["return_status"].isna()
)

return_status,False,True
order_status,,
Cancelled,0,8398
Completed,0,113559
Pending,0,6697
Returned,9462,0


In [ ]:
pd.crosstab(
    df["order_status"],
    df["customer_rating"].isna()
)

customer_rating,False,True
order_status,,
Cancelled,0,8398
Completed,113559,0
Pending,0,6697
Returned,0,9462


### Customer Ratings — Missing Values

Customer ratings are available for all 113,559 completed orders, while ratings are missing for all cancelled, pending, and returned orders.

This indicates that customer ratings are recorded only for completed orders in this dataset.

**Analytical limitation:** Average customer ratings represent completed orders only and should not be interpreted as overall customer satisfaction across all order statuses.

In [ ]:
df[
    ["customer_rating", "review_sentiment", "customer_review"]].isna().sum(axis=1).value_counts()


0    113559
3     24557
Name: count, dtype: int64

In [ ]:
df["coupon_code"].value_counts(dropna=False)

coupon_code
NaN       110502
SAVE39       731
SAVE49       727
SAVE40       719
SAVE12       716
SAVE33       715
SAVE26       712
SAVE25       711
SAVE15       709
SAVE36       709
SAVE46       707
SAVE44       704
SAVE21       704
SAVE27       702
SAVE22       702
SAVE17       701
SAVE13       701
SAVE18       699
SAVE43       699
SAVE31       693
SAVE19       693
SAVE42       691
SAVE20       690
SAVE32       687
SAVE41       685
SAVE11       683
SAVE48       682
SAVE38       681
SAVE37       681
SAVE34       679
SAVE16       677
SAVE35       675
SAVE23       674
SAVE28       674
SAVE47       673
SAVE30       670
SAVE45       665
SAVE14       662
SAVE24       656
SAVE29       641
SAVE10       634
Name: count, dtype: int64

In [ ]:
df.loc[
    df["coupon_code"].isna(),
    "discount_amount"
].describe()

count    110502.000000
mean        238.071515
std         379.741538
min           0.000000
25%          26.532500
50%         103.285000
75%         276.800000
max        6759.020000
Name: discount_amount, dtype: float64

In [ ]:
df.loc[
    df["coupon_code"].isna(),
    "discount_amount"
].gt(0).value_counts()

discount_amount
True     96266
False    14236
Name: count, dtype: int64

In [ ]:
df["campaign_name"].value_counts(dropna=False)

campaign_name
NaN                   83233
Default_Campaign      22989
Google_Search_Q1       2748
Google_Shopping        2746
Google_Display         2731
FB_Retargeting         2270
FB_Dynamic             2233
Friend_Invite          2232
Referral_Program       2198
FB_Prospecting         2161
Insta_Influencer       1822
Insta_Reels            1799
Insta_Stories          1773
Abandoned_Cart         1520
Weekly_Newsletter      1459
Promo_Email            1459
Influencer_Program     1404
Affiliate_Partner      1339
Name: count, dtype: int64

In [ ]:
df["order_id"].duplicated().sum()

np.int64(0)

In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df["customer_age"].describe()

count    138116.000000
mean         45.862681
std          16.467676
min          18.000000
25%          32.000000
50%          46.000000
75%          60.000000
max          74.000000
Name: customer_age, dtype: float64

In [ ]:
df["customer_id"].nunique()

24911

In [ ]:
unique_customers = df.drop_duplicates(
    subset = "customer_id"
)
unique_customers["customer_age"].describe()

count    24911.000000
mean        45.916944
std         16.450201
min         18.000000
25%         32.000000
50%         46.000000
75%         60.000000
max         74.000000
Name: customer_age, dtype: float64

In [ ]:
df.groupby("customer_id")["customer_age"].nunique().gt(1).sum()

np.int64(0)

In [ ]:
df["order_date"] = pd.to_datetime(df["order_date"])

In [ ]:
df["order_date"].agg(["min", "max"])

min   2021-01-01
max   2025-12-31
Name: order_date, dtype: datetime64[us]

In [ ]:
df["order_date"].isna().sum()

np.int64(0)

In [ ]:
df["order_date"].dt.year.value_counts().sort_index()

order_date
2021    27554
2022    27564
2023    27632
2024    27768
2025    27598
Name: count, dtype: int64

The number of orders remained relatively stable between 2021 and 2025, averaging approximately 27,600 orders annually. No substantial upward or downward trend was observed in annual order volume.

In [ ]:
df["order_date"].dt.month.value_counts().sort_index()

order_date
1      9693
2      8027
3     10822
4     10120
5     10238
6     10880
7     10207
8     12064
9     10855
10    11582
11    16061
12    17567
Name: count, dtype: int64

In [ ]:
monthly_orders = df.groupby([
    df["order_date"].dt.year,
    df["order_date"].dt.month
]).size().unstack()

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

monthly_orders

order_date,1,2,3,4,5,6,7,8,9,10,11,12
order_date,,,,,,,,,,,,
2021,1906,1565,2166,1943,2096,2203,2036,2474,2134,2324,3139,3568
2022,1934,1615,2126,2008,2041,2174,2011,2349,2124,2333,3257,3592
2023,1926,1585,2134,2040,2054,2189,2010,2484,2190,2347,3177,3496
2024,1965,1617,2202,2123,2048,2176,2029,2394,2181,2293,3297,3443
2025,1962,1645,2194,2006,1999,2138,2121,2363,2226,2285,3191,3468


### Order Date Analysis — Key Findings

The dataset contains 138,116 orders covering the period from January 1, 2021, to December 31, 2025.

- No missing or duplicate order IDs were identified.
- Annual order volume remained relatively stable, with approximately 27,600 orders per year.
- December consistently recorded the highest number of orders, followed by November.
- February had the lowest monthly order count in each year, although its shorter duration should be considered.
- The seasonal pattern was consistent across all five years.

**Observation:** The dataset demonstrates stable annual order volume and recurring seasonal fluctuations, with particularly high order activity during November and December.

**Limitation:** Order volume alone does not indicate revenue growth or profitability. Financial metrics require separate analysis.

## Financial Data Validation

**Objective:** Examine the distribution and consistency of financial variables, identify potential anomalies, and verify whether the financial metrics are suitable for further sales and profitability analysis.

In [ ]:
df[[
    "gross_sales",
    "net_sales",
    "profit",
    "discount_amount"
]].describe()

,gross_sales,net_sales,profit,discount_amount
count,138116.000000,138116.000000,138116.000000,138116.000000
mean,1375.384176,1282.503575,551.322046,237.621614
std,1291.009554,1169.274669,516.300936,378.779459
min,6.310000,5.800000,-1281.590000,0.000000
25%,458.920000,448.200000,190.757500,26.327500
50%,995.730000,950.760000,408.395000,103.285000
75%,1871.200000,1744.582500,752.845000,277.125000
max,16051.190000,16218.000000,7083.680000,6759.020000


In [ ]:
(df["profit"] < 0).sum()

np.int64(1296)

In [ ]:
loss_orders = df[df["profit"] < 0]

loss_orders [[
    "order_id",
    "gross_sales",
    "discount_amount",
    "net_sales",
    "product_cost",
    "shipping_cost",
    "profit"
]].head(10)

,order_id,gross_sales,discount_amount,net_sales,product_cost,shipping_cost,profit
1,ORD-773460,3144.64,1456.06,2018.41,2031.88,9.00,-22.47
149,ORD-567393,5558.21,2943.84,2826.85,2886.87,29.48,-89.50
156,ORD-613985,3350.18,1683.01,2011.70,2202.05,11.09,-201.44
158,ORD-324161,259.71,111.35,161.97,170.91,3.22,-12.16
163,ORD-828591,4076.70,2150.22,2298.28,2533.00,5.77,-240.49
266,ORD-459603,665.22,399.13,296.68,331.44,11.96,-46.72
552,ORD-685844,1646.60,675.84,1060.25,1044.25,21.54,-5.54
773,ORD-590144,1751.91,768.89,1199.62,1198.53,20.00,-18.91
1078,ORD-887807,3307.56,1502.36,1931.74,1955.84,36.28,-60.38
1086,ORD-990629,2294.60,1267.92,1101.93,1275.35,3.38,-176.80


### Negative Profitability — Initial Findings

The dataset contains 1,296 orders with negative profit, representing approximately 0.94% of all orders.

An initial examination of 10 unprofitable orders revealed that several had substantial discounts. For example, the first five orders had discount rates ranging from approximately 43% to 53% of gross sales.

**Hypothesis:** High discount rates may be associated with negative order profitability.

**Next Step:** Compare discount rates between profitable and unprofitable orders to determine whether higher discounts are consistently associated with financial losses.

In [ ]:
calculated_profit = (
    df["net_sales"] - df["product_cost"] - df["shipping_cost"]
)
profit_difference = (
    df["profit"] - calculated_profit
)

profit_difference.abs().describe()

count    1.381160e+05
mean     8.828892e-14
std      1.382045e-13
min      0.000000e+00
25%      0.000000e+00
50%      2.842171e-14
75%      1.136868e-13
max      2.273737e-12
dtype: float64

In [ ]:
(profit_difference.abs() > 0.01).sum()

np.int64(0)

In [ ]:
df["discount_rate"] = (
    df["discount_amount"] / df["gross_sales"] * 100
)
df['discount_rate'].describe()

count    138116.000000
mean         15.117085
std          12.287188
min           0.000000
25%           6.543998
50%          12.747787
75%          20.598256
max          60.019940
Name: discount_rate, dtype: float64

In [ ]:
df.groupby(
    df["profit"] < 0
)["discount_rate"].agg(["count", "mean", "median"])

,count,mean,median
profit,,,
False,136820,14.770556,12.644910
True,1296,51.700560,52.459488


### Discount Rates and Order Profitability

A comparison of profitable and unprofitable orders revealed a substantial difference in discount rates.

- **Unprofitable orders (1,296):** Average discount rate of 51.70%, with a median of 52.46%.
- **Non-loss orders (136,820):** Average discount rate of 14.77%, with a median of 12.64%.

**Key Finding:** Orders with negative profits had significantly higher discount rates, with an average difference of approximately 36.93 percentage points.

**Interpretation:** High discount rates are strongly associated with negative order profitability. However, product costs and shipping expenses must also be considered before attributing losses solely to discounts.

In [ ]:
high_discount = df[df["discount_rate"] > 40]
(high_discount["profit"] < 0).mean()*100

np.float64(17.08041014570966)

### High Discounts and Risk of Financial Loss

To further investigate the relationship between discounts and profitability, orders with discount rates above 40% were analyzed.

**Key Findings:**
- Overall, 0.94% of all orders were unprofitable.
- Among orders with discounts above 40%, 17.08% were unprofitable.
- The proportion of unprofitable orders was approximately 18 times higher in the high-discount group compared to the overall dataset.
- However, 82.92% of orders with discounts above 40% remained profitable or broke even.

**Interpretation:** Higher discount rates are strongly associated with an increased risk of negative profitability.

**Next Step:** Analyze loss rates across different discount ranges to identify where the risk of financial loss increases most significantly.

In [ ]:
df["discount_group"] = pd.cut(
    df["discount_rate"],
    bins=[0,10,20,30,40,50,61],
    labels=["0-10%", "10-20%", "20-30%", "30-40%", "40-50%", "50-60%"],
    include_lowest=True

)

In [ ]:
df.groupby("discount_group", observed=True)["profit"].agg(
    orders = "count",
    loss_rate = lambda x: (x<0).mean() * 100
)

,orders,loss_rate
discount_group,,
0-10%,52895,0.000000
10-20%,49198,0.000000
20-30%,17729,0.000000
30-40%,10882,0.275685
40-50%,5683,8.622207
50-60%,1729,44.881434


### Profitability Risk by Discount Range

To identify potential profitability risks, orders were grouped into discount ranges and the percentage of unprofitable orders was calculated for each group.

**Key Findings:**
- No unprofitable orders were observed at discount rates of 30% or below.
- At discount rates between 30% and 40%, the loss rate was only 0.28%.
- The loss rate increased to 8.62% for discounts between 40% and 50%.
- At discount rates above 50%, approximately 44.88% of orders were unprofitable.

**Conclusion:** The analysis reveals a strong association between higher discount rates and negative profitability, particularly when discounts exceed 40–50%.

**Business Recommendation:** Consider reviewing high-discount pricing strategies and introducing profitability checks before approving discounts above 40%, with particular attention to discounts exceeding 50%.

**Limitation:** These findings demonstrate an association, not proof that discounts alone cause losses. Product costs, shipping expenses, and data limitations should also be considered.

### Additional Hypotheses — Factors Contributing to Negative Profit

Although high discount rates are strongly associated with negative profitability, other factors may contribute to financial losses.

**Hypotheses:**

1. **High Product Costs:** Higher product costs may reduce profitability.
2. **Low Initial Profit Margins:** Orders with a small difference between gross sales and product costs may become unprofitable even with moderate discounts.
3. **High Shipping Costs:** Higher shipping expenses relative to sales may contribute to negative profits.

**Next Step:** Compare cost-to-sales ratios, initial profit margins, and shipping costs between profitable and unprofitable orders to evaluate these hypotheses.

In [ ]:
df["initial_margin"] = (
    (df["gross_sales"] - df["product_cost"])
    / df["gross_sales"] * 100
)

In [ ]:
df.groupby(
    df["profit"] < 0
)["initial_margin"].agg(["count", "mean", "median"])

,count,mean,median
profit,,,
False,136820,51.454588,51.426079
True,1296,41.210751,40.436198


### Initial Margin and Order Profitability

The analysis examined whether lower initial margins are associated with negative order profitability.

**Key Findings:**
- Non-loss orders had an average initial margin of 51.45%.
- Unprofitable orders had an average initial margin of 41.21%.
- The difference between the two groups was approximately 10.24 percentage points.
- Unprofitable orders also had substantially higher average discount rates (51.70% compared with 14.77%).

**Interpretation:** Lower initial margins, combined with higher discount rates, may contribute to negative order profitability by reducing the margin available to cover costs.

**Next Step:** Examine the relationship between initial margins, discount rates, and shipping expenses at the individual order level.

In [ ]:
df["discount_exceeds_margin"] = (
    df["discount_rate"] > df["initial_margin"]
)

pd.crosstab(
    df["discount_exceeds_margin"],
    df["profit"] < 0
)

profit,False,True
discount_exceeds_margin,,
False,135586,0
True,1234,1296


### Discount Rates Exceeding Initial Margins

The analysis compared discount rates with initial profit margins to examine their relationship with negative profitability.

**Key Findings:**
- All 1,296 unprofitable orders had discount rates exceeding their initial margins.
- Among the 2,530 orders where discounts exceeded initial margins, 51.23% were unprofitable.
- The remaining 48.77% of these orders maintained non-negative profits.

**Interpretation:** Discount rates exceeding initial margins are strongly associated with financial losses. However, this condition does not always result in negative profitability.

**Next Step:** Investigate how net sales are calculated and examine the impact of shipping expenses on profitability.

In [ ]:
calculated_net_sales = (
    df["gross_sales"] - 
    df["discount_amount"] + df["tax_amount"]
)
net_sales_difference = (
    df["net_sales"] - calculated_net_sales
)
net_sales_difference.abs().describe()


count    138116.000000
mean         22.278080
std          16.022739
min           0.000000
25%           9.820000
50%          19.190000
75%          30.750000
max         149.520000
dtype: float64

In [ ]:
calculated_net_sales_2 = (
    df["gross_sales"] - 
    df["discount_amount"] + df["tax_amount"] + df["shipping_cost"]
)
difference_2 = (
    df["net_sales"] - calculated_net_sales_2
)
difference_2.abs().describe()

count    1.381160e+05
mean     1.158977e-13
std      2.037568e-13
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      2.273737e-13
max      3.637979e-12
dtype: float64

### Net Sales Calculation Validation

The following formula was identified and validated across all 138,116 orders:

**Net Sales = Gross Sales − Discount Amount + Tax Amount + Shipping Cost**

The calculated values matched the recorded `net_sales`, with a maximum absolute difference of approximately 0.00000000000364, attributable to floating-point precision.

**Key Finding:** Shipping costs and taxes are included in the dataset's calculation of net sales.

**Important Limitation:** This definition differs from conventional accounting treatment of net sales. In addition, shipping costs are added to net sales and subsequently deducted when calculating profit, effectively cancelling their impact on the recorded profit.

**Conclusion:** Although the financial calculations are internally consistent, the definitions of net sales and profit should be treated as dataset-specific rather than standard accounting measures.

In [ ]:
df["effective_tax_rate"] = (
    df["tax_amount"] / (df["gross_sales"] - df["discount_amount"]) * 100
)

df.groupby("customer_country")["effective_tax_rate"].agg(["min", "mean", "max"]).round(2)

,min,mean,max
customer_country,,,
Australia,9.96,10.0,10.04
Canada,12.95,13.0,13.02
Germany,18.95,19.0,19.05
India,17.97,18.0,18.05
UAE,4.97,5.0,5.03
UK,19.95,20.0,20.04
USA,6.94,7.0,7.10


### Tax Rate Validation and Financial Data Limitations

The dataset includes transactions from seven countries with different currencies and tax systems.

Effective tax rates were calculated using the following formula:

**Effective Tax Rate = Tax Amount / (Gross Sales − Discount Amount) × 100**

**Key Findings:**
- Tax rates are nearly constant within each country.
- The dataset applies approximately 7% for the USA, 20% for the UK, 19% for Germany, 13% for Canada, 10% for Australia, 18% for India, and 5% for the UAE.
- This suggests that the dataset uses simplified country-level tax assumptions rather than accounting for regional differences, product-specific rates, or exemptions.

**Financial Limitations:**
- Monetary values are recorded in multiple currencies and cannot be directly aggregated without conversion.
- The dataset's profit formula includes tax amounts as a positive component, which may overstate profitability compared with conventional accounting treatment.
- Cross-country profitability comparisons may therefore be misleading.

**Analytical Decision:** Monetary metrics will be analyzed separately by currency. Reported profit will be treated as a dataset-specific metric rather than verified accounting profit.

In [ ]:
df["adjusted_profit"] = (
    df["profit"] - df["tax_amount"]
)

df[["profit", "adjusted_profit"]].describe()

,profit,adjusted_profit
count,138116.000000,138116.000000
mean,551.322046,428.859114
std,516.300936,409.477691
min,-1281.590000,-1655.640000
25%,190.757500,145.727500
50%,408.395000,319.665000
75%,752.845000,590.650000
max,7083.680000,5151.440000


In [ ]:
print("Original unprofitable orders: ")
print((df["profit"] < 0).sum())

print("Adjusted unprofitable orders: ")
print((df["adjusted_profit"] < 0). sum())

Original unprofitable orders: 
1296
Adjusted unprofitable orders: 
2530


### Impact of Tax Adjustment on Order Profitability

The dataset's original profit calculation includes tax amounts as a positive component. To assess the impact of this assumption, an adjusted profit metric was calculated by excluding taxes.

**Key Findings:**
- Original unprofitable orders: 1,296 (0.94%).
- Adjusted unprofitable orders: 2,530 (1.83%).
- An additional 1,234 orders became unprofitable after excluding taxes.
- The number of unprofitable orders increased by approximately 95.2%.

**Interpretation:** Including tax amounts in the original profit calculation substantially reduces the reported number of unprofitable orders.

The adjusted profit formula also reveals that orders become unprofitable when the discount amount exceeds the difference between gross sales and product costs.

**Limitation:** Adjusted profit is a simplified analytical measure and does not represent fully verified accounting profit.

In [ ]:
df.groupby("discount_group", observed=True)["adjusted_profit"].agg(
    orders= "count",
    loss_rate=lambda x: (x< 0).mean()*100
)

,orders,loss_rate
discount_group,,
0-10%,52895,0.000000
10-20%,49198,0.000000
20-30%,17729,0.000000
30-40%,10882,2.223856
40-50%,5683,20.904452
50-60%,1729,63.620590


### Profitability Risk After Tax Adjustment

The analysis compared loss rates across discount ranges using both the original and tax-adjusted profit metrics.

**Key Findings:**
- No losses were observed at discount rates of 30% or below under either profit definition.
- For discounts between 30% and 40%, the adjusted loss rate increased from 0.28% to 2.22%.
- For discounts between 40% and 50%, the adjusted loss rate increased from 8.62% to 20.90%.
- For discounts between 50% and 60%, the adjusted loss rate increased from 44.88% to 63.62%.

**Conclusion:** Excluding tax amounts reveals a substantially greater share of unprofitable orders, particularly in higher discount ranges.

**Business Implication:** Discount approval decisions should account for the available margin before discounting, rather than relying solely on a fixed discount percentage.

**Limitation:** Adjusted profit is a simplified metric based on the dataset's financial structure and does not include all potential business expenses.

In [ ]:
order_item = pd.read_csv("order_items.csv")
products = pd.read_csv("product_catalog.csv")

In [ ]:
print("Order Items: ", order_item.shape)
print("Product Catalog: ", products.shape)

Order Items:  (397569, 12)
Product Catalog:  (1175, 9)


In [ ]:
print(order_item.columns.tolist())

print(products.columns.tolist())

['order_id', 'product_id', 'quantity', 'unit_price', 'discount_percentage', 'discount_amount', 'gross_sales', 'tax_amount', 'shipping_cost', 'net_sales', 'product_cost', 'profit']
['product_id', 'product_name', 'product_category', 'product_subcategory', 'brand', 'supplier', 'unit_price', 'product_cost', 'product_rating']


In [ ]:
products["product_id"].duplicated().sum()

np.int64(0)

In [ ]:
product_sales = order_item.merge(
    products[
        ["product_id", "product_name", "product_category", "product_subcategory"]
    ],

    on="product_id",
    how="left",
    validate="many_to_one"
)

In [ ]:
print("Before merge: ", order_item.shape)
print("After merge: ", product_sales.shape)

product_sales[
    ["order_id", "product_name", "product_category", "profit"]
].head()

Before merge:  (397569, 12)
After merge:  (397569, 15)


,order_id,product_name,product_category,profit
0,ORD-301242,Garmin Total client-driven task-force Medical ...,Health & Wellness,172.12
1,ORD-301242,Tommy Hilfiger Open-architected full-range foc...,Fashion,172.92
2,ORD-773460,Swarovski Re-engineered discrete monitoring Ea...,Jewelry,-65.84
3,ORD-773460,NARS Enterprise-wide actuating firmware Fragra...,Beauty & Personal Care,43.37
4,ORD-449374,Mars Multi-tiered homogeneous artificial intel...,Grocery,12.53


In [ ]:
item_profit = product_sales.groupby("order_id")["profit"].sum()

order_profit = df.set_index("order_id")["profit"]

profit_check = item_profit - order_profit

profit_check.abs().describe()

count    138116.000000
mean         47.708489
std         222.639554
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max        5181.370000
Name: profit, dtype: float64

### Data Quality Issue — Profit Inconsistencies Between Tables

To validate financial data consistency, I compared the total profit calculated from individual order items with the profit recorded in the main orders dataset.

**Key Findings:**
- The median difference was 0, indicating that profit values matched for at least half of the orders.
- The average absolute difference was 47.71.
- The maximum absolute difference reached 5,181.37.
- Approximately 8.13% of orders had profit discrepancies greater than 0.01.

**Issue Identified:** Profit values in the `order_items` table do not consistently match the corresponding values in the main orders dataset.

**Potential Impact:** These inconsistencies may affect product-level profitability analysis and lead to inaccurate financial conclusions when aggregating profit by product category.

**Next Step:** Investigate the affected orders to identify possible causes of the discrepancies before proceeding with product profitability analysis.

In [ ]:
profit_comparison = pd.DataFrame({
    "order_profit": order_profit,
    "item_profit": item_profit
})

profit_comparison["difference_2"] = (
    profit_comparison["item_profit"] - 
    profit_comparison["order_profit"]
)

profit_comparison[
    profit_comparison["difference_2"].abs() > 0.01
].head(10)

,order_profit,item_profit,difference_2
order_id,,,
ORD-100046,59.73,349.16,289.43
ORD-100101,92.39,659.87,567.48
ORD-100338,1633.37,2267.64,634.27
ORD-100432,295.01,524.24,229.23
ORD-100439,138.44,402.68,264.24
ORD-100510,79.52,532.75,453.23
ORD-100564,682.26,739.18,56.92
ORD-100746,765.59,1028.39,262.80
ORD-100758,1056.48,1499.11,442.63


In [ ]:
item_gross = product_sales.groupby("order_id")["gross_sales"].sum()

order_gross = df.set_index("order_id")["gross_sales"]

gross_difference = item_gross - order_gross

gross_difference.abs().describe()

count    138116.000000
mean        118.604197
std         557.947990
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max       12266.970000
Name: gross_sales, dtype: float64

In [ ]:
print("Items Gross > Order Gross: ")
print((gross_difference > 0.01).sum())

print("Item Gross < Order Gross: ")
print((gross_difference < -0.01).sum())

print("Matching Gross Sales: ")
print((gross_difference.abs() <= 0.01).sum())

Items Gross > Order Gross: 
11226
Item Gross < Order Gross: 
0
Matching Gross Sales: 
126890


In [ ]:
item_profit = product_sales.groupby("order_id")["profit"].sum()

order_profit = df.set_index("order_id")["profit"]

profit_difference = item_profit - order_profit

In [ ]:
pd.crosstab(
    gross_difference.abs() > 0.01,
    profit_difference.abs() > 0.01,

    rownames=["Gross Sales Mismatch"],
    colnames=["Profit Mismatch"]
)

Profit Mismatch,False,True
Gross Sales Mismatch,,
False,126890,0
True,0,11226


In [ ]:
# Rebuild comparison before analyzing order-item counts
orders_summary = df.set_index("order_id")[["gross_sales", "profit"]]
items_summary = product_sales.groupby("order_id").agg(
    items_gross=("gross_sales", "sum"),
    items_profit=("profit", "sum")
)
comparison = orders_summary.join(items_summary)
comparison["gross_mismatch"] = (
    comparison["items_gross"] - comparison["gross_sales"]
).abs() > 0.01
comparison["profit_mismatch"] = (
    comparison["items_profit"] - comparison["profit"]
).abs() > 0.01
item_counts = product_sales.groupby("order_id").size()
comparison["item_count"] = item_counts
comparison.groupby("gross_mismatch")["item_count"].describe()

,count,mean,std,min,25%,50%,75%,max
gross_mismatch,,,,,,,,
False,126890.0,2.650430,1.237177,1.0,2.0,2.0,3.0,5.0
True,11226.0,5.456619,1.895943,2.0,4.0,5.0,7.0,18.0


In [ ]:
product_sales.loc[
    product_sales["order_id"] == "ORD-100046",
    ["product_id", "quantity", "unit_price", "gross_sales", "profit"]
].to_string(index=False)

' product_id  quantity  unit_price  gross_sales  profit\nPROD-000269         2       69.38       138.76   59.73\nPROD-000029         1      604.28       604.28  289.43'

In [ ]:
product_sales.loc[
    product_sales["order_id"] == "ORD-100101",
    ["product_id", "quantity", "gross_sales", "profit"]
]

,product_id,quantity,gross_sales,profit
73923,PROD-001024,3,207.66,92.39
234855,PROD-000890,2,466.60,165.24
234856,PROD-000440,1,372.51,242.82
234857,PROD-000097,1,300.58,159.42


In [ ]:
item_calculated_profit = (
    product_sales["net_sales"] - 
    product_sales["product_cost"] - 
    product_sales["shipping_cost"]
)

item_profit_difference = (
    product_sales["profit"] - item_calculated_profit
)

(item_profit_difference.abs() > 0.01).sum()

np.int64(0)

In [ ]:
profit_validation = product_sales[[
    "order_id", "net_sales",
    "product_cost", "shipping_cost",
    "profit"
]].copy()

profit_validation["calculated_profit"] = item_calculated_profit

profit_validation["difference_2"] = item_profit_difference

profit_validation.head(20)

,order_id,net_sales,product_cost,shipping_cost,profit,calculated_profit,difference_2
0,ORD-301242,351.88,173.28,6.48,172.12,172.12,0.000000e+00
1,ORD-301242,593.52,415.05,5.55,172.92,172.92,2.842171e-14
2,ORD-773460,1919.32,1976.16,9.00,-65.84,-65.84,1.421085e-13
3,ORD-773460,99.09,55.72,0.00,43.37,43.37,-7.105427e-15
4,ORD-449374,19.42,5.91,0.98,12.53,12.53,-1.776357e-15
5,ORD-449374,448.93,251.82,12.07,185.04,185.04,-2.842171e-14
6,ORD-567636,301.14,172.77,6.37,122.00,122.00,2.842171e-14
7,ORD-567636,245.38,105.07,14.48,125.83,125.83,0.000000e+00
8,ORD-820028,2639.24,1179.60,9.52,1450.12,1450.12,0.000000e+00
9,ORD-820028,73.92,22.89,9.65,41.38,41.38,0.000000e+00


In [ ]:
item_calculated_profit = (
    product_sales["net_sales"] - 
    product_sales["product_cost"] - 
    product_sales["shipping_cost"]
)

item_profit_difference = (
    product_sales["profit"] - 
    item_calculated_profit
)

print("Rows checked: ", len(product_sales))

print(
    "Profit mismatches: ",
    (item_profit_difference.abs() > 0.01).sum()
)

Rows checked:  397569
Profit mismatches:  0


### Product-Level Profit Validation

The profit calculation was validated across all 397,569 order-item records.

**Formula:** Profit = Net Sales − Product Cost − Shipping Cost

**Key Findings:**
- No discrepancies greater than 0.01 were identified in product-level profit calculations.
- Profit calculations are internally consistent within both the order-level and item-level datasets.
- However, 11,226 orders (8.13%) have inconsistencies between their order-level and aggregated item-level financial values.

**Conclusion:** The identified discrepancies appear to be related to differences between the datasets rather than incorrect product-level profit calculations.

**Next Step:** Investigate whether additional order-item records explain the discrepancies.

In [ ]:
product_sales["cumulative_profit"] = (
    product_sales.groupby("order_id")["profit"].cumsum()
)

In [ ]:
product_sales.loc[
    product_sales["order_id"] == "ORD-100101",
    ["product_id", "profit", "cumulative_profit"]
]

,product_id,profit,cumulative_profit
73923,PROD-001024,92.39,92.39
234855,PROD-000890,165.24,257.63
234856,PROD-000440,242.82,500.45
234857,PROD-000097,159.42,659.87


In [ ]:
order_profit_lookup = df.set_index("order_id")["profit"]

product_sales["recorded_order_profit"] = (
    product_sales["order_id"].map(order_profit_lookup)
)

product_sales["cumulative_match"] = ((
    product_sales["cumulative_profit"] - 
    product_sales["recorded_order_profit"]
).abs() <= 0.01)

problem_ids = profit_difference[
    profit_difference.abs() > 0.01
].index

print("Problem orders: ", len(problem_ids))

Problem orders:  11226


In [ ]:
matches = (
    product_sales[
        product_sales["order_id"].isin(problem_ids)
    ].groupby("order_id")["cumulative_match"].any()
)

print(matches.value_counts())

cumulative_match
True    11226
Name: count, dtype: int64


In [ ]:
product_sales["cumulative_gross"] = (
    product_sales.groupby("order_id")["gross_sales"].cumsum()

)

order_gross_lookup = df.set_index("order_id")["gross_sales"]

product_sales["recorded_order_gross"] = (
    product_sales["order_id"].map(order_gross_lookup)
)

product_sales["both_match"] = (
    (product_sales["cumulative_profit"] - 
     product_sales["recorded_order_profit"]).abs() <= 0.01
)& (
   ( product_sales["cumulative_gross"] - 
    product_sales["recorded_order_gross"]).abs() <= 0.01
)
both_matches = (
    product_sales[
        product_sales["order_id"].isin(problem_ids)
    ].groupby("order_id")["both_match"].any()
)

print(both_matches.value_counts())

both_match
True    11226
Name: count, dtype: int64


### Investigation of Financial Discrepancies Between Orders and Order Items

To investigate the financial inconsistencies between the main orders dataset and the `order_items` table, I compared cumulative item-level gross sales and profit with the corresponding order-level values.

**Key Findings:**
- 11,226 orders (8.13%) showed discrepancies in both gross sales and profit.
- In all affected orders, the total gross sales from `order_items` exceeded the values recorded in the main orders dataset.
- Orders with discrepancies contained more product line items on average (5.46 compared with 2.65).
- For all 11,226 affected orders, the cumulative gross sales and profit of the first several order items matched the corresponding order-level values.

**Interpretation:** These findings strongly suggest that the main orders dataset includes financial totals for only a subset of the product line items recorded in `order_items`.

**Potential Impact:** Including all product line items in financial aggregations may result in higher sales and profit totals than those reported in the main orders dataset.

**Next Step:** Identify and quantify the additional product line items associated with these discrepancies.

**Data Handling Decision:** No records will be removed or modified until the cause of the inconsistencies is better understood.

### Quantifying Additional Order Items

**Objective:** Determine how many product line items are included in the `order_items` table but are not reflected in the financial totals of the main orders dataset.

The previous analysis showed that cumulative gross sales and profit matched the recorded order totals at a specific item position for all 11,226 affected orders.

**Next Steps:**
1. Identify the position where cumulative financial values match the order-level totals.
2. Count the total number of product line items in each affected order.
3. Calculate the number of additional items beyond the matching position.
4. Examine the distribution of additional items across affected orders.

**Purpose:** Quantify the extent of the inconsistencies and assess their potential impact on product-level financial analysis.

In [ ]:
problem_items = product_sales[
    product_sales["order_id"].isin(problem_ids)
].copy()

problem_items["item_position"] = (
    problem_items.groupby("order_id").cumcount() + 1
)
matched_position = (
    problem_items[problem_items["both_match"]]
    .groupby("order_id")["item_position"]
    .first()
)

total_positions = (
    problem_items.groupby("order_id").size()
)

extra_positions = total_positions - matched_position

extra_positions.describe()

count    11226.000000
mean         2.814092
std          1.422223
min          1.000000
25%          2.000000
50%          3.000000
75%          4.000000
max         13.000000
dtype: float64

In [ ]:
extra_positions.sum()

np.int64(31591)

In [ ]:
extra_positions.sum() / len(product_sales) * 100

np.float64(7.946042070684585)

In [ ]:
int(extra_positions.sum())

31591

In [ ]:
extra_items = problem_items[
    problem_items["item_position"] >
    problem_items["order_id"].map(matched_position)
].copy()

extra_items["currency"] = extra_items["order_id"].map(
    df.set_index("order_id")["currency"]
)

extra_items.groupby("currency").agg(
    extra_items=("product_id", "size"),
    gross_sales=("gross_sales", "sum"),
    profit=("profit", "sum")
).round(2)

,extra_items,gross_sales,profit
currency,,,
AED,958,512075.22,204471.81
AUD,1493,757927.19,297906.84
CAD,1683,904037.36,362221.22
EUR,2626,1359950.25,546010.53
GBP,4537,2333924.83,949823.94
INR,1312,696834.82,279073.11
USD,18982,9816387.54,3912135.34


### Financial Impact of Additional Order Items

The reconciliation analysis identified 31,591 additional product line items across 11,226 orders (8.13% of all orders).

These items appeared beyond the point at which cumulative item-level gross sales and profit matched the financial totals recorded in the main orders dataset.

**Key Findings:**
- Additional line items represent approximately 7.95% of all records in `order_items`.
- For USD-denominated orders, these items account for approximately $9.82 million in gross sales and $3.91 million in recorded profit.
- Financial discrepancies were identified across all seven currencies in the dataset.

**Interpretation:** The differences between order-level and item-level records have a substantial impact on aggregated financial metrics.

**Data Quality Concern:** Combining all item-level financial records without addressing these inconsistencies may overstate sales and profit relative to the main orders dataset.

**Next Step:** Establish a consistent analytical scope for product-level profitability analysis while preserving and documenting the original records.

**Limitation:** The analysis identifies additional records relative to order-level totals but does not establish whether those records are erroneous or represent valid transactions.

In [ ]:
product_sales["item_position"] = (
    product_sales.groupby("order_id").cumcount() +1
)

product_sales["matching_position"] = (
    product_sales["order_id"].map(matched_position)
)

In [ ]:
reconciled_item = product_sales[
    product_sales["matching_position"].isna()
    |
    (
        product_sales["item_position"] <=
        product_sales["matching_position"]
    )
].copy()


In [ ]:
print("Original items: ", len(product_sales))
print("Reconciled items: ", len(reconciled_item))
print(
    "Excluded items: ",
    len(product_sales) - len(reconciled_item)
)

Original items:  397569
Reconciled items:  365978
Excluded items:  31591


In [ ]:
reconciled_totals = reconciled_item.groupby("order_id").agg(
    items_gross = ("gross_sales", "sum"),
    items_profit = ("profit", "sum")
)

order_totals = df.set_index("order_id")[["gross_sales", "profit"]]

validation = order_totals.join(reconciled_totals)
validation["gross_difference"]=(
    validation["items_gross"] - validation["gross_sales"]
)

validation["profit_difference"] = (
    validation ["items_profit"] - validation["profit"]
)
print("Gross Sales mismatches: ",
      (validation["gross_difference"]. abs() > 0.01).sum())

print("Profit mismatches: ", 
      (validation["profit_difference"].abs() > 0.01).sum())

print("Orders with missing item totals: ",
      validation["items_gross"].isna().sum())

Gross Sales mismatches:  0
Profit mismatches:  0
Orders with missing item totals:  0


### Validation of Reconciled Order Items

After identifying 31,591 additional line items, I created a reconciled dataset containing 365,978 product line items.

To validate the dataset, I aggregated gross sales and profit by `order_id` and compared the results with the original order-level financial totals.

**Validation Results:**
- Gross sales mismatches: 0
- Profit mismatches: 0
- Orders with missing item totals: 0
- All 138,116 orders passed the reconciliation checks within a tolerance of 0.01.

**Conclusion:** The reconciled dataset is financially consistent with the main orders dataset and will be used for the following product profitability analysis.

**Limitation:** The excluded line items were preserved in the original dataset. Their exclusion is a methodological decision based on financial consistency, not proof that those records are erroneous.

Which product categories generate the highest gross sales?

Which product categories generate the highest profit?

Which product categories have the highest profit margins?

In [ ]:
reconciled_item["country"] = reconciled_item["order_id"].map(
    df.set_index("order_id")["customer_country"]
)

reconciled_item["currency"] = reconciled_item ["order_id"].map(
    df.set_index("order_id")["currency"]
)

In [ ]:
us_items = reconciled_item[
    (reconciled_item["country"] == "USA") &
    (reconciled_item["currency"] == "USD")
].copy()

print("US product line items: ", len(us_items))
print("Product categories: ", us_items["product_category"].nunique())
print("Missing categories: ", us_items["product_category"].isna().sum())

US product line items:  218797
Product categories:  15
Missing categories:  0


### Product Category Overview — US Market

Before analyzing sales and profitability, I examined the product categories represented in the US market.

**Objective:** Identify the 15 product categories included in the dataset and understand the distribution of product line items across categories.

This overview provides context for the subsequent analysis of gross sales, total profit, and profit margins.

In [ ]:
us_items["product_category"].value_counts()

product_category
Sports & Outdoors         16800
Books & Media             16270
Health & Wellness         16001
Beauty & Personal Care    15357
Automotive                15214
Office Supplies           15045
Grocery                   14598
Toys & Games              14555
Home Appliances           14507
Electronics               14505
Baby & Kids               14000
Pet Supplies              13654
Home & Kitchen            12900
Jewelry                   12771
Fashion                   12620
Name: count, dtype: int64

**Initial Observations:**

The US market dataset contains 15 product categories, with a relatively balanced distribution of product line items.

- Sports & Outdoors has the highest number of line items (16,800), followed by Books & Media (16,270) and Health & Wellness (16,001).
- Fashion has the lowest number of line items (12,620).
- No single category strongly dominates the distribution.

However, the frequency of product line items does not necessarily indicate higher sales revenue or profitability. Further financial analysis is required.

RQ1: Which product categories generate the highest gross sales?

In [ ]:
category_sales = (
    us_items.groupby("product_category")["gross_sales"]
    .sum()
    .sort_values(ascending=False)
)
category_sales.round(2)

product_category
Electronics               24426640.57
Jewelry                   15194847.09
Home Appliances           14885521.12
Automotive                 9765169.74
Sports & Outdoors          9239701.60
Home & Kitchen             6457118.55
Baby & Kids                5687936.83
Health & Wellness          5408464.67
Office Supplies            5251087.77
Fashion                    4347394.00
Pet Supplies               3708868.37
Toys & Games               3170704.10
Beauty & Personal Care     2680698.03
Grocery                    1669404.70
Books & Media              1557291.76
Name: gross_sales, dtype: float64

### RQ1: Which Product Categories Generate the Highest Gross Sales?

Gross sales were aggregated by product category using reconciled order items from the US market.

**Key Findings:**
- Electronics generated the highest gross sales ($24.43M).
- Jewelry ($15.19M) and Home Appliances ($14.89M) ranked second and third.
- Books & Media recorded the lowest gross sales ($1.56M), despite having the second-highest number of product line items.
- Electronics ranked only tenth by line item count but first by gross sales.

**Interpretation:**

The number of product line items does not necessarily correspond to gross sales performance. Categories with fewer line items may generate substantially higher sales, potentially due to differences in product prices and quantities sold.

Further analysis of unit prices and quantities could help explain these differences.

Which product categories generate the highest profit?

In [ ]:
category_profit = (
    us_items.groupby("product_category")["profit"]
    .sum()
    .sort_values(ascending=False)
)
category_profit.round(2)

product_category
Electronics               6999437.39
Home Appliances           5216793.54
Jewelry                   5215657.63
Sports & Outdoors         3654506.00
Automotive                3279681.72
Home & Kitchen            2655971.73
Baby & Kids               2450912.99
Office Supplies           2387865.45
Health & Wellness         2191438.94
Fashion                   1868290.91
Pet Supplies              1660197.56
Toys & Games              1450566.18
Beauty & Personal Care    1264609.23
Grocery                    860397.78
Books & Media              766938.85
Name: profit, dtype: float64

### RQ2: Which Product Categories Generate the Highest Profit?

Total recorded profit was aggregated by product category using reconciled order items from the US market.

**Key Findings:**
- Electronics generated the highest total profit ($7.00M).
- Home Appliances ($5.22M) and Jewelry ($5.22M) ranked second and third, respectively.
- Home Appliances generated slightly more profit than Jewelry despite having lower gross sales.
- Sports & Outdoors generated more profit than Automotive despite having lower gross sales.
- Books & Media recorded the lowest total profit ($0.77M).

**Interpretation:**

Higher gross sales do not always translate into higher total profit. Differences in product costs, discounts, and other financial factors may influence profitability across categories.

Further analysis of profit margins is needed to compare profitability relative to sales volume.

Which product categories have the highest profit margins?

In [ ]:
category_summary = pd.DataFrame({
    "gross_sales": category_sales,
    "total_profit": category_profit
})

category_summary["profit_margin"] = (
    category_summary["total_profit"] /
    category_summary["gross_sales"] * 100
)

category_summary.sort_values(
    "profit_margin", ascending=False
).round(2)

,gross_sales,total_profit,profit_margin
product_category,,,
Grocery,1669404.70,860397.78,51.54
Books & Media,1557291.76,766938.85,49.25
Beauty & Personal Care,2680698.03,1264609.23,47.17
Toys & Games,3170704.10,1450566.18,45.75
Office Supplies,5251087.77,2387865.45,45.47
Pet Supplies,3708868.37,1660197.56,44.76
Baby & Kids,5687936.83,2450912.99,43.09
Fashion,4347394.00,1868290.91,42.97
Home & Kitchen,6457118.55,2655971.73,41.13


### RQ3: Which Product Categories Have the Highest Profit Margins?

Profit margins were calculated as total recorded profit divided by total gross sales for each product category in the US market.

**Key Findings:**
- Grocery had the highest profit margin (51.54%), followed by Books & Media (49.25%) and Beauty & Personal Care (47.17%).
- Electronics had the lowest profit margin (28.65%), despite generating the highest gross sales and total profit.
- Books & Media ranked last in both gross sales and total profit but second in profit margin.
- Categories with higher total sales generally did not lead in relative profitability.

**Interpretation:**

The results highlight a distinction between absolute profitability and relative profitability. High-revenue categories such as Electronics generate substantial total profit but retain a smaller percentage of gross sales as recorded profit.

Meanwhile, lower-revenue categories such as Grocery and Books & Media have higher profit margins but contribute less total profit.

**Conclusion:** Both total profit and profit margin should be considered when evaluating product category performance. Neither metric alone provides a complete picture of profitability.

RQ4: How do product costs differ across product categories?

In [ ]:
category_cost = us_items.groupby("product_category").agg(
    gross_sales = ("gross_sales", "sum"),
    product_cost = ("product_cost", "sum")
)
category_cost["cost_ratio"] = (
    category_cost["product_cost"] /
    category_cost["gross_sales"] * 100
)
category_cost.sort_values(
    "cost_ratio", ascending=False
).round(2)

,gross_sales,product_cost,cost_ratio
product_category,,,
Electronics,24426640.57,14689270.33,60.14
Automotive,9765169.74,5373253.87,55.02
Jewelry,15194847.09,8228074.67,54.15
Home Appliances,14885521.12,7914485.88,53.17
Sports & Outdoors,9239701.60,4507156.86,48.78
Health & Wellness,5408464.67,2595574.18,47.99
Home & Kitchen,6457118.55,3055448.17,47.32
Fashion,4347394.00,1965792.74,45.22
Baby & Kids,5687936.83,2559072.37,44.99


### RQ4: How Do Product Costs Differ Across Product Categories?

To investigate differences in profitability, I calculated the product cost ratio for each category by dividing total product costs by total gross sales.

**Key Findings:**
- Electronics had the highest product cost ratio (60.14%) and the lowest recorded profit margin (28.65%).
- Grocery had the lowest product cost ratio (37.07%) and the highest recorded profit margin (51.54%).
- The difference in product cost ratios between Electronics and Grocery was 23.07 percentage points.
- The corresponding difference in profit margins was 22.89 percentage points.

**Interpretation:**

The results suggest that product costs are a major contributor to differences in profit margins across categories.

Higher product costs relative to gross sales leave less room for profit, while categories with lower cost ratios tend to retain a greater share of sales as recorded profit.

However, discounts, shipping costs, and the dataset's tax treatment may also influence these results.

RQ5: How do discount rates vary across product categories?

In [ ]:
category_discount = us_items.groupby("product_category").agg(
    gross_sales=("gross_sales", "sum"),
    total_discount=("discount_amount", "sum")
)

category_discount["discount_rate"] = (
    category_discount["total_discount"] /
    category_discount["gross_sales"] * 100
)

category_discount.sort_values(
    "discount_rate", ascending=False
).round(2)

,gross_sales,total_discount,discount_rate
product_category,,,
Baby & Kids,5687936.83,1005707.42,17.68
Fashion,4347394.00,764139.15,17.58
Home Appliances,14885521.12,2613298.09,17.56
Pet Supplies,3708868.37,650877.09,17.55
Beauty & Personal Care,2680698.03,468006.59,17.46
Sports & Outdoors,9239701.60,1611980.63,17.45
Office Supplies,5251087.77,910341.30,17.34
Home & Kitchen,6457118.55,1119343.55,17.34
Jewelry,15194847.09,2630610.80,17.31


### RQ5: How Do Discount Rates Vary Across Product Categories?

To investigate whether discounts contribute to differences in profitability, I calculated the total discount amount as a percentage of gross sales for each product category.

**Key Findings:**
- Discount rates were relatively consistent across all 15 product categories, ranging from 17.02% to 17.68%.
- Baby & Kids had the highest discount rate (17.68%).
- Electronics had the lowest discount rate (17.02%), despite having the lowest profit margin (28.65%).
- Grocery had a discount rate of 17.19%, only slightly higher than Electronics.

**Interpretation:**

The results do not support the hypothesis that higher category-level discount rates explain the lower profit margin of Electronics.

Instead, product cost differences appear to be a more important factor. Electronics had the highest product cost ratio (60.14%), while Grocery had the lowest (37.07%).

**Conclusion:** Product cost structure appears to explain much more of the observed variation in recorded profit margins than differences in aggregate discount rates. Additional factors, including shipping costs and tax treatment, may also contribute.

### RQ6: How Do Product Markups Vary Across Categories?

**Hypothesis:** Categories with lower recorded profit margins may have lower selling-price markups relative to product costs.

To investigate this, I will compare category-level markup percentages and pre-discount gross margins.

This analysis will help assess whether differences in pricing relative to product costs contribute to the observed profitability patterns.

In [ ]:
category_markup = us_items.groupby("product_category").agg(
    gross_sales=("gross_sales", "sum"),
    product_cost=("product_cost", "sum")
)

category_markup["markup_pct"] = (
    (category_markup["gross_sales"] - category_markup["product_cost"]) / 
    category_markup["product_cost"] * 100
)
category_markup["pre_discount_margin_pct"] = (
    (category_markup["gross_sales"] - category_markup["product_cost"]) / 
     category_markup["gross_sales"] * 100
)

category_markup.sort_values(
    "markup_pct", ascending=False
).round(2)

,gross_sales,product_cost,markup_pct,pre_discount_margin_pct
product_category,,,,
Grocery,1669404.70,618833.82,169.77,62.93
Books & Media,1557291.76,615389.16,153.06,60.48
Beauty & Personal Care,2680698.03,1102971.23,143.04,58.86
Toys & Games,3170704.10,1355222.95,133.96,57.26
Office Supplies,5251087.77,2256733.42,132.69,57.02
Pet Supplies,3708868.37,1611854.04,130.10,56.54
Baby & Kids,5687936.83,2559072.37,122.27,55.01
Fashion,4347394.00,1965792.74,121.15,54.78
Home & Kitchen,6457118.55,3055448.17,111.33,52.68


### RQ6: How Do Product Markups Vary Across Categories?

To further investigate differences in profitability, I calculated category-level markups relative to product costs and pre-discount gross margins.

**Key Findings:**
- Grocery had the highest aggregated markup (169.77%), followed by Books & Media (153.06%).
- Electronics had the lowest markup (66.29%) and the lowest pre-discount gross margin (39.86%).
- Grocery's markup was approximately 2.56 times that of Electronics.
- Categories with higher markups generally had higher recorded profit margins.

**Interpretation:**

The results suggest that differences in selling prices relative to product costs are closely associated with category-level profitability.

Electronics generated the highest total profit due to its substantial sales volume but retained a smaller proportion of gross sales as profit. Grocery generated much lower total profit but had a substantially higher markup relative to product costs.

Markup and product cost ratio describe the same underlying relationship from different perspectives and should not be treated as independent profitability drivers.

**Conclusion:** Differences in category-level price-to-cost relationships appear to be the primary explanation for the observed variation in recorded profit margins, while aggregate discount rates remain relatively similar across categories.

## Business Opportunity Assessment

### RQ7: Which Product Categories Show Sales Growth Over Time?

**Business Objective:** Identify product categories with growing or declining sales to evaluate potential opportunities for assortment expansion.

**Approach:** Analyze annual gross sales by product category in the US market from 2021 to 2025.

**Key Questions:**
- Which categories show consistent sales growth?
- Which categories experienced declining sales?
- Do categories with high current profitability also demonstrate sales growth?

In [ ]:
us_items["order_date"] = us_items["order_id"].map(
    df.set_index("order_id")["order_date"]
)

us_items["order_date"] = pd.to_datetime(
    us_items["order_date"]
)
us_items["year"] = us_items["order_date"].dt.year

In [ ]:
annual_category_sales = (
    us_items.groupby(["year", "product_category"])["gross_sales"]
    .sum()
    .unstack()
)

annual_category_sales.round(2)

product_category,Automotive,Baby & Kids,Beauty & Personal Care,Books & Media,Electronics,Fashion,Grocery,Health & Wellness,Home & Kitchen,Home Appliances,Jewelry,Office Supplies,Pet Supplies,Sports & Outdoors,Toys & Games
year,,,,,,,,,,,,,,,
2021,1942813.71,1155697.21,548771.50,295737.79,4885341.55,876865.02,335314.48,1095552.24,1329134.14,2989418.93,2974656.45,1056643.72,715733.05,1856511.55,648284.39
2022,1941227.58,1193431.21,544490.27,302474.57,4956502.52,881880.61,337952.65,1099423.31,1254455.01,2934977.67,2943933.16,1031138.06,711285.81,1869457.95,641435.26
2023,1976974.34,1099878.44,541960.11,315272.36,4926112.12,852434.13,328818.64,1040091.20,1304779.81,2999917.18,3181733.97,1070289.22,749636.98,1866990.81,633451.08
2024,1935222.82,1152170.58,517007.04,317489.57,4779048.66,852709.62,334945.93,1096706.37,1253457.61,3079667.71,3110215.04,1057397.75,786511.08,1892971.25,626135.31
2025,1968931.29,1086759.39,528469.11,326317.47,4879635.72,883504.62,332373.00,1076691.55,1315291.98,2881539.63,2984308.47,1035619.02,745701.45,1753770.04,621398.06


In [ ]:
sales_growth = pd.DataFrame({
    "sales_2021": annual_category_sales.loc[2021],
    "sales_2025": annual_category_sales.loc[2025]
})

sales_growth["growth_pct"] = (
    (sales_growth["sales_2025"] - sales_growth["sales_2021"]) /
    sales_growth["sales_2021"] * 100
)

sales_growth.sort_values(
    "growth_pct", ascending=False
).round(2)

,sales_2021,sales_2025,growth_pct
product_category,,,
Books & Media,295737.79,326317.47,10.34
Pet Supplies,715733.05,745701.45,4.19
Automotive,1942813.71,1968931.29,1.34
Fashion,876865.02,883504.62,0.76
Jewelry,2974656.45,2984308.47,0.32
Electronics,4885341.55,4879635.72,-0.12
Grocery,335314.48,332373.00,-0.88
Home & Kitchen,1329134.14,1315291.98,-1.04
Health & Wellness,1095552.24,1076691.55,-1.72


### RQ7: Which Product Categories Show Sales Growth Over Time?

I compared annual gross sales between 2021 and 2025 for all 15 product categories in the US market.

**Key Findings:**
- Five categories recorded an increase in gross sales, while ten experienced a decrease.
- Books & Media had the highest growth (+10.34%), followed by Pet Supplies (+4.19%).
- Baby & Kids experienced the largest decline (−5.97%).
- Electronics remained relatively stable (−0.12%).
- Grocery, which had the highest recorded profit margin, experienced a slight decrease in sales (−0.88%).

**Business Interpretation:**

Books & Media stands out as a potential category for further investigation because it combines relatively high profitability with positive sales growth.

However, changes in gross sales alone do not establish an increase in customer demand. Further analysis of quantities sold and year-by-year trends is required before making assortment expansion recommendations.

**Next Step:** Examine annual sales trends and product quantities to distinguish potential demand growth from changes in sales value.

RQ8: Which Product Categories Show Growth in Units Sold?

In [ ]:
annual_units = (
    us_items.groupby(["year", "product_category"])["quantity"]
    .sum()
    .unstack()
)
annual_units.round(0)

product_category,Automotive,Baby & Kids,Beauty & Personal Care,Books & Media,Electronics,Fashion,Grocery,Health & Wellness,Home & Kitchen,Home Appliances,Jewelry,Office Supplies,Pet Supplies,Sports & Outdoors,Toys & Games
year,,,,,,,,,,,,,,,
2021,6398,6008,6574,6556,6060,5409,6239,6891,5672,6269,5263,6467,5712,7170,6284
2022,6389,6126,6634,6801,6275,5416,6238,6888,5385,6122,5305,6306,5536,7074,6194
2023,6454,5748,6566,6939,6173,5266,5978,6588,5486,6232,5565,6461,5777,7259,6170
2024,6280,5951,6312,7011,5997,5251,6206,7016,5256,6414,5505,6352,6026,7277,6107
2025,6525,5825,6559,7159,5989,5445,6074,6708,5551,5895,5267,6267,5848,6774,6096


In [ ]:
units_growth = pd.DataFrame({
    "units_2021": annual_units.loc[2021],
    "units_2025": annual_units.loc[2025]
})

units_growth["units_growth_pct"] = (
    (units_growth["units_2025"] - units_growth["units_2021"]) / 
    units_growth["units_2021"] * 100
)

units_growth.sort_values(
    "units_growth_pct", ascending=False
).round(2)

,units_2021,units_2025,units_growth_pct
product_category,,,
Books & Media,6556,7159,9.20
Pet Supplies,5712,5848,2.38
Automotive,6398,6525,1.98
Fashion,5409,5445,0.67
Jewelry,5263,5267,0.08
Beauty & Personal Care,6574,6559,-0.23
Electronics,6060,5989,-1.17
Home & Kitchen,5672,5551,-2.13
Grocery,6239,6074,-2.64


### RQ8: Which Product Categories Show Growth in Units Sold?

To investigate whether changes in gross sales reflect changes in sales volume, I compared the total number of units sold in 2021 and 2025 across all 15 product categories.

**Key Findings:**
- Books & Media had the highest increase in units sold (+9.20%), accompanied by a 10.34% increase in gross sales.
- Pet Supplies showed positive growth in both units sold (+2.38%) and gross sales (+4.19%).
- Automotive experienced a 1.98% increase in units sold.
- Grocery experienced a 2.64% decline in units sold despite having the highest recorded profit margin (51.54%).
- Home Appliances experienced the largest decline in units sold (−5.97%).

**Business Interpretation:**

Books & Media and Pet Supplies stand out as potential candidates for further investigation because both categories experienced growth in sales value and units sold.

However, changes between 2021 and 2025 do not necessarily indicate consistent year-over-year growth. Annual trends should be examined before drawing conclusions about sustained demand.

**Next Step:** Evaluate the consistency of annual sales trends before developing recommendations for assortment expansion.

In [ ]:
selected_categories = [
    "Books & Media",
    "Pet Supplies",
    "Grocery"
]
annual_units[selected_categories]

product_category,Books & Media,Pet Supplies,Grocery
year,,,
2021,6556,5712,6239
2022,6801,5536,6238
2023,6939,5777,5978
2024,7011,6026,6206
2025,7159,5848,6074


### Annual Sales Volume Trends: Selected Categories

To assess whether sales growth was consistent over time, I compared annual units sold for Books & Media, Pet Supplies, and Grocery from 2021 to 2025.

**Key Findings:**
- Books & Media demonstrated consistent year-over-year growth in units sold, increasing from 6,556 units in 2021 to 7,159 units in 2025 (+9.20%).
- Pet Supplies experienced fluctuations, with overall growth of 2.38% between 2021 and 2025.
- Grocery showed no consistent upward trend, with units sold declining by 2.64% over the period.

**Business Interpretation:**

Books & Media stands out as a potential candidate for assortment expansion because it combines consistent growth in sales volume with a relatively high recorded profit margin (49.25%).

Pet Supplies may also warrant further investigation, although its sales volume growth was less consistent.

Despite having the highest recorded profit margin (51.54%), Grocery has not demonstrated sustained growth in units sold.

**Preliminary Recommendation:** Prioritize Books & Media for further evaluation before making assortment expansion decisions.

In [ ]:
annual_profit = (
    us_items.groupby(["year", "product_category"])["profit"]
    .sum()
    .unstack()
)
annual_profit[
    ["Books & Media", "Pet Supplies", "Grocery"]
].round(2)

product_category,Books & Media,Pet Supplies,Grocery
year,,,
2021,145662.83,326208.86,173878.95
2022,147655.76,319910.09,173387.07
2023,156950.09,332331.41,171641.88
2024,155339.01,345133.77,171541.62
2025,161331.16,336613.43,169948.26


### Annual Profit Trends and Business Opportunity Assessment

To evaluate whether positive sales trends were accompanied by profitability growth, I compared annual recorded profit for Books & Media, Pet Supplies, and Grocery from 2021 to 2025.

**Key Findings:**
- Books & Media recorded a 10.76% increase in profit, alongside 10.34% growth in gross sales and 9.20% growth in units sold.
- Pet Supplies recorded 3.19% profit growth, although annual results fluctuated.
- Grocery experienced a 2.26% decline in profit despite having the highest overall recorded profit margin (51.54%).
- Books & Media showed positive profit growth over the full period, although profit declined slightly in 2024.

**Business Interpretation:**

Books & Media demonstrates the strongest combination of sales growth, unit growth, and profit growth among the three categories examined.

Pet Supplies also shows positive overall trends, but with less consistent annual performance.

Grocery remains highly profitable relative to gross sales, but its declining sales volume and profit suggest that expansion should not be recommended based on margin alone.

**Preliminary Recommendation:**

Prioritize Books & Media for further evaluation as a potential assortment expansion opportunity. Consider Pet Supplies as a secondary candidate.

These recommendations are preliminary and do not account for inventory requirements, additional operating costs, or future demand uncertainty.

In [ ]:
annual_financials = us_items.groupby(
    ["year", "product_category"]
).agg(
    gross_sales=("gross_sales", "sum"),
    profit=("profit", "sum")
)

annual_financials["profit_margin"] = (
    annual_financials["profit"] / 
    annual_financials["gross_sales"] * 100
)

annual_margins = annual_financials["profit_margin"].unstack()

annual_margins [ 
    ["Books & Media", "Pet Supplies", "Grocery"]
].round(2)

product_category,Books & Media,Pet Supplies,Grocery
year,,,
2021,49.25,45.58,51.86
2022,48.82,44.98,51.31
2023,49.78,44.33,52.20
2024,48.93,43.88,51.21
2025,49.44,45.14,51.13


### RQ9: How Has Profit Margin Changed Over Time?

To evaluate the consistency of profitability, I analyzed annual recorded profit margins for Books & Media, Pet Supplies, and Grocery from 2021 to 2025.

**Key Findings:**
- Books & Media maintained a relatively stable profit margin, ranging from 48.82% to 49.78%.
- Books & Media's profit margin increased slightly from 49.25% in 2021 to 49.44% in 2025.
- Pet Supplies experienced some fluctuations, with its margin decreasing from 45.58% to 45.14%.
- Grocery maintained the highest profit margin among the three categories but experienced a slight decline from 51.86% to 51.13%.

**Business Interpretation:**

Books & Media demonstrated growth in gross sales, units sold, and total profit while maintaining a stable recorded profit margin.

This combination makes Books & Media a promising candidate for further assortment expansion evaluation.

Pet Supplies also shows potential, although its annual performance was less consistent. Grocery remains attractive in terms of relative profitability but has not demonstrated positive sales volume growth.

**Preliminary Recommendation:**

Prioritize Books & Media for further evaluation, with Pet Supplies as a secondary candidate. Additional operational and inventory data would be needed before making an actual investment decision.

## Business Performance Summary

**Objective:** Consolidate the key financial and growth metrics for all 15 product categories in the US market.

This summary will support the comparison of category performance and help identify potential opportunities for profitable assortment expansion.

**Metrics:**
- Gross Sales — Total sales before discounts
- Total Profit — Total recorded profit
- Profit Margin — Recorded profit as a percentage of gross sales
- Sales Growth — Percentage change in gross sales from 2021 to 2025
- Units Growth — Percentage change in units sold from 2021 to 2025

In [ ]:
business_summary = pd.DataFrame({
    "gross_sales": category_sales,
    "total_profit": category_profit,
    "sales_growth_pct": sales_growth["growth_pct"],
    "units_growth_pct": units_growth["units_growth_pct"]
})

business_summary["profit_margin_pct"] = (
    business_summary["total_profit"] / 
    business_summary["gross_sales"] * 100
)

business_summary = business_summary [[
    "gross_sales",
    "total_profit",
    "profit_margin_pct",
    "sales_growth_pct",
    "units_growth_pct"
]]
business_summary = business_summary.sort_values(
    "total_profit",
    ascending=False

)
business_summary.round(2)

,gross_sales,total_profit,profit_margin_pct,sales_growth_pct,units_growth_pct
product_category,,,,,
Electronics,24426640.57,6999437.39,28.65,-0.12,-1.17
Home Appliances,14885521.12,5216793.54,35.05,-3.61,-5.97
Jewelry,15194847.09,5215657.63,34.33,0.32,0.08
Sports & Outdoors,9239701.60,3654506.00,39.55,-5.53,-5.52
Automotive,9765169.74,3279681.72,33.59,1.34,1.98
Home & Kitchen,6457118.55,2655971.73,41.13,-1.04,-2.13
Baby & Kids,5687936.83,2450912.99,43.09,-5.97,-3.05
Office Supplies,5251087.77,2387865.45,45.47,-1.99,-3.09
Health & Wellness,5408464.67,2191438.94,40.52,-1.72,-2.66


In [ ]:
print("Number of categories:", len(business_summary))

print("\nMissing values:")
print(business_summary.isna().sum())

Number of categories: 15

Missing values:
gross_sales          0
total_profit         0
profit_margin_pct    0
sales_growth_pct     0
units_growth_pct     0
dtype: int64


In [ ]:
business_summary.to_csv(
    "business_category_summary_US.csv",
    index=True,
    index_label = "product_category"

)

In [ ]:
powerbi_data = us_items [
    [
        "order_id",
        "order_date",
        "year",
        "product_id",
        "product_name",
        "product_category",
        "quantity",
        "gross_sales",
        "discount_amount",
        "product_cost",
        "shipping_cost",
        "profit"
    ]
].copy()

print(powerbi_data.shape)

(218797, 12)


In [ ]:
powerbi_data.to_csv("powerbi_us_sales.csv", index=False)
print("Power BI export completed:", len(powerbi_data), "rows")

Power BI export completed: 218797 rows
